In [1]:
# ============================================================
# CELL 1 — IMPORT LIBRARIES
# ============================================================

import re
import pandas as pd
import matplotlib.pyplot as plt

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ============================================================
# CELL 2 — LOAD THE UCI SMS SPAM COLLECTION DATASET
# ============================================================

# Path to the original UCI dataset
file_path = "../dataset/SMSSpamCollection"

# Read the file line-by-line.
# We split only at the FIRST tab character so that valid
# messages containing additional tab characters are not lost.
records = []

with open(file_path, "r", encoding="utf-8") as file:
    for line in file:
        line = line.rstrip("\n\r")

        # Ignore completely empty lines
        if not line.strip():
            continue

        # Split only at the first TAB
        parts = line.split("\t", 1)

        # A valid record must contain both label and message
        if len(parts) == 2:
            label, message = parts
            records.append([label, message])

# Create the Pandas DataFrame
df = pd.DataFrame(
    records,
    columns=["label", "message"]
)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

print("\nFirst 5 records:")
display(df.head())

Dataset loaded successfully.
Dataset shape: (5574, 2)

First 5 records:


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [3]:
# ============================================================
# CELL 3 — INITIAL DATASET VALIDATION
# ============================================================

print("========== INITIAL DATASET VALIDATION ==========\n")

# Display basic dataset dimensions
print("Number of rows   :", len(df))
print("Number of columns:", len(df.columns))

# Display column names
print("\nColumn names:")
print(df.columns.tolist())

# Display data types
print("\nData types:")
print(df.dtypes)

# Check for missing values
print("\nMissing values:")
print(df.isnull().sum())

# Check the unique target labels
print("\nUnique labels:")
print(df["label"].unique())

# Display the number of messages in each class
print("\nClass distribution:")
print(df["label"].value_counts())

# Display detailed DataFrame information
print("\nDataset information:")
df.info()

========== INITIAL DATASET VALIDATION ==========

Number of rows   : 5574
Number of columns: 2

Column names:
['label', 'message']

Data types:
label      str
message    str
dtype: object

Missing values:
label      0
message    0
dtype: int64

Unique labels:
<StringArray>
['ham', 'spam']
Length: 2, dtype: str

Class distribution:
label
ham     4827
spam     747
Name: count, dtype: int64

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 5574 entries, 0 to 5573
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   label    5574 non-null   str  
 1   message  5574 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB


In [4]:
# ============================================================
# CELL 4 — CHECK EXACT DUPLICATES AND DATA QUALITY
# ============================================================

print("========== DATA QUALITY CHECK ==========\n")

# Count completely identical rows
duplicate_rows = df.duplicated().sum()

# Count unique complete rows
unique_rows = df.drop_duplicates().shape[0]

# Count unique original message texts
unique_messages = df["message"].nunique()

print("Total records          :", len(df))
print("Exact duplicate rows   :", duplicate_rows)
print("Unique complete rows   :", unique_rows)
print("Unique message texts   :", unique_messages)

# Check for missing values
print("\nMissing values:")
print(df.isnull().sum())

# Verify that all labels belong to the expected classes
valid_labels = {"ham", "spam"}
invalid_labels = set(df["label"].unique()) - valid_labels

print("\nInvalid labels:", invalid_labels)

if not invalid_labels:
    print("All labels are valid: ham / spam")
else:
    print("WARNING: Invalid labels detected.")

========== DATA QUALITY CHECK ==========

Total records          : 5574
Exact duplicate rows   : 403
Unique complete rows   : 5171
Unique message texts   : 5171

Missing values:
label      0
message    0
dtype: int64

Invalid labels: set()
All labels are valid: ham / spam


In [5]:
# ============================================================
# CELL 5 — REMOVE EXACT DUPLICATE RECORDS
# ============================================================

# Create a separate working DataFrame.
# The original 'df' remains unchanged.
clean_df = df.copy()

# Remove completely identical records.
clean_df = (
    clean_df
    .drop_duplicates()
    .reset_index(drop=True)
)

print("========== EXACT DUPLICATE REMOVAL ==========\n")

print("Records before removal :", len(df))
print("Records after removal  :", len(clean_df))
print("Rows removed           :", len(df) - len(clean_df))

# Verify that no exact duplicate rows remain
print("\nRemaining exact duplicate rows:")
print(clean_df.duplicated().sum())

# Display the class distribution after duplicate removal
print("\nClass distribution after duplicate removal:")
print(clean_df["label"].value_counts())

# Display the current dataset shape
print("\nCurrent dataset shape:")
print(clean_df.shape)

========== EXACT DUPLICATE REMOVAL ==========

Records before removal : 5574
Records after removal  : 5171
Rows removed           : 403

Remaining exact duplicate rows:
0

Class distribution after duplicate removal:
label
ham     4518
spam     653
Name: count, dtype: int64

Current dataset shape:
(5171, 2)


In [37]:
# ============================================================
# CELL 6 — DEFINE TEXT PREPROCESSING FUNCTION
# ============================================================

def preprocess_text(text):
    """
    Normalize an SMS message before TF-IDF vectorization.

    Operations:
    1. Convert text to lowercase.
    2. Convert common emoticons into meaningful text tokens.
    3. Preserve letters and numbers.
    4. Remove unnecessary punctuation and symbols.
    5. Normalize multiple spaces.
    """

    # Convert text to lowercase
    text = text.lower()

    # Convert common emoticons into a text token
    text = re.sub(
        r"(?::|;|=)(?:-)?(?:\)|d)",
        " smile ",
        text
    )

    # Keep English letters, numbers, and whitespace
    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    # Normalize whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    # Remove leading and trailing whitespace
    text = text.strip()

    return text


# ------------------------------------------------------------
# TEST THE PREPROCESSING FUNCTION
# ------------------------------------------------------------

test_messages = [
    "Congratulations!!! You WON $1000. Claim NOW!!!",
    "Hey, are you coming tomorrow?",
    "FREE entry!!! Call 9876543210 NOW!!!",
    ":)",
    ":-) :-)"
]

print("========== PREPROCESSING FUNCTION TEST ==========\n")

for message in test_messages:

    cleaned_message = preprocess_text(message)

    print("Original:")
    print(message)

    print("\nCleaned:")
    print(cleaned_message)

    print("-" * 60)

========== PREPROCESSING FUNCTION TEST ==========

Original:
Congratulations!!! You WON $1000. Claim NOW!!!

Cleaned:
congratulations you won 1000 claim now
------------------------------------------------------------
Original:
Hey, are you coming tomorrow?

Cleaned:
hey are you coming tomorrow
------------------------------------------------------------
Original:
FREE entry!!! Call 9876543210 NOW!!!

Cleaned:
free entry call 9876543210 now
------------------------------------------------------------
Original:
:)

Cleaned:
smile
------------------------------------------------------------
Original:
:-) :-)

Cleaned:
smile smile
------------------------------------------------------------


In [38]:
# ============================================================
# CELL 7 — APPLY TEXT PREPROCESSING TO THE DATASET
# ============================================================

# Apply the preprocessing function to every SMS message
# and store the result in a separate column.
clean_df["clean_message"] = clean_df["message"].apply(
    preprocess_text
)

print("========== TEXT PREPROCESSING COMPLETED ==========\n")

print("Number of messages processed:", len(clean_df))

# Display original and cleaned messages side-by-side
print("\nSample of original vs cleaned messages:")

display(
    clean_df[
        ["label", "message", "clean_message"]
    ].head(10)
)

# Check whether any message became empty
empty_messages = (
    clean_df["clean_message"].str.len() == 0
).sum()

print("\nEmpty cleaned messages:", empty_messages)

if empty_messages == 0:
    print("No messages became empty after preprocessing.")
else:
    print("WARNING: Some messages became empty.")

========== TEXT PREPROCESSING COMPLETED ==========

Number of messages processed: 5131

Sample of original vs cleaned messages:


,label,message,clean_message
0,ham,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...
1,ham,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...
3,ham,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah i don t think he goes to usf he lives arou...
5,spam,FreeMsg Hey there darling it's been 3 week's n...,freemsg hey there darling it s been 3 week s n...
6,ham,Even my brother is not like to speak with me. ...,even my brother is not like to speak with me t...
7,ham,As per your request 'Melle Melle (Oru Minnamin...,as per your request melle melle oru minnaminun...
8,spam,WINNER!! As a valued network customer you have...,winner as a valued network customer you have b...
9,spam,Had your mobile 11 months or more? U R entitle...,had your mobile 11 months or more u r entitled...



Empty cleaned messages: 0
No messages became empty after preprocessing.


In [39]:
# ============================================================
# CELL 8 — INVESTIGATE EMPTY CLEANED MESSAGES
# ============================================================

# Identify messages that became empty after preprocessing
empty_mask = clean_df["clean_message"].str.len() == 0

empty_records = clean_df.loc[
    empty_mask,
    ["label", "message", "clean_message"]
]

print("========== EMPTY MESSAGE INVESTIGATION ==========\n")

print("Number of empty cleaned messages:", len(empty_records))

print("\nOriginal messages that became empty:")

display(empty_records)

========== EMPTY MESSAGE INVESTIGATION ==========

Number of empty cleaned messages: 0

Original messages that became empty:


,label,message,clean_message


In [40]:
# ============================================================
# CELL 9 — IMPROVED TEXT PREPROCESSING FUNCTION
# ============================================================

def preprocess_text(text):
    """
    Normalize an SMS message before TF-IDF vectorization.

    Operations:
    1. Convert text to lowercase.
    2. Convert common emoticons into meaningful text tokens.
    3. Preserve letters and numbers.
    4. Remove unnecessary punctuation and symbols.
    5. Normalize multiple spaces.
    """

    # Convert text to lowercase
    text = text.lower()

    # --------------------------------------------------------
    # Convert common positive/neutral emoticons to text.
    # This prevents messages containing only emoticons from
    # becoming empty after punctuation removal.
    # --------------------------------------------------------
    
    text = re.sub(
        r"(?::|;|=)(?:-)?(?:\)|d)",
        " smile ",
        text
    )

    # Keep English letters, numbers, and whitespace.
    # Numbers are intentionally preserved because they may
    # provide useful information for spam classification.
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Replace multiple whitespace characters with one space
    text = re.sub(r"\s+", " ", text)

    # Remove leading and trailing whitespace
    text = text.strip()

    return text


# ------------------------------------------------------------
# TEST THE IMPROVED FUNCTION
# ------------------------------------------------------------

test_messages = [
    "Congratulations!!! You WON $1000. Claim NOW!!!",
    "Hey, are you coming tomorrow?",
    "FREE entry!!! Call 9876543210 NOW!!!",
    ":)",
    ":-) :-)"
]

print("========== IMPROVED PREPROCESSING TEST ==========\n")

for message in test_messages:

    cleaned_message = preprocess_text(message)

    print("Original:")
    print(message)

    print("\nCleaned:")
    print(cleaned_message)

    print("-" * 60)

========== IMPROVED PREPROCESSING TEST ==========

Original:
Congratulations!!! You WON $1000. Claim NOW!!!

Cleaned:
congratulations you won 1000 claim now
------------------------------------------------------------
Original:
Hey, are you coming tomorrow?

Cleaned:
hey are you coming tomorrow
------------------------------------------------------------
Original:
FREE entry!!! Call 9876543210 NOW!!!

Cleaned:
free entry call 9876543210 now
------------------------------------------------------------
Original:
:)

Cleaned:
smile
------------------------------------------------------------
Original:
:-) :-)

Cleaned:
smile smile
------------------------------------------------------------


In [41]:
    # ============================================================
# CELL 10 — REAPPLY IMPROVED PREPROCESSING
# ============================================================

# Apply the updated preprocessing function to every original
# SMS message in the working dataset.
clean_df["clean_message"] = clean_df["message"].apply(
    preprocess_text
)

print("========== REPROCESSING COMPLETED ==========\n")

print("Number of messages processed:", len(clean_df))

# Check whether any message is empty after preprocessing
empty_mask = clean_df["clean_message"].str.len() == 0
empty_count = empty_mask.sum()

print("Empty cleaned messages:", empty_count)

if empty_count == 0:
    print("All messages have valid cleaned text.")
else:
    print("WARNING: Empty cleaned messages still exist.")

# Verify the two messages that previously became empty
print("\nPreviously problematic messages:")

display(
    clean_df.loc[
        clean_df["message"].isin([":)", ":-) :-)"]),
        ["label", "message", "clean_message"]
    ]
)

========== REPROCESSING COMPLETED ==========

Number of messages processed: 5131
Empty cleaned messages: 0
All messages have valid cleaned text.

Previously problematic messages:


,label,message,clean_message
4472,ham,:-) :-),smile smile


In [42]:
# ============================================================
# CELL 11 — CHECK NORMALIZED MESSAGE DUPLICATES
# ============================================================

# After preprocessing, different original messages can become
# identical. For example:
#
# "Ok no prob"
# "Ok no prob..."
#
# both become:
# "ok no prob"
#
# We must identify these before creating the train/test split.

print("========== NORMALIZED MESSAGE DUPLICATE CHECK ==========\n")

# Count duplicate cleaned messages
normalized_duplicate_count = (
    clean_df["clean_message"].duplicated().sum()
)

print(
    "Duplicate normalized messages:",
    normalized_duplicate_count
)

# Find messages that have multiple occurrences
duplicate_mask = clean_df["clean_message"].duplicated(
    keep=False
)

duplicate_records = clean_df.loc[
    duplicate_mask,
    ["label", "message", "clean_message"]
].sort_values("clean_message")

# Display examples if duplicates exist
if normalized_duplicate_count > 0:
    print("\nExamples of normalized duplicates:")
    display(duplicate_records.head(20))
else:
    print("\nNo normalized duplicate messages found.")

========== NORMALIZED MESSAGE DUPLICATE CHECK ==========

Duplicate normalized messages: 0

No normalized duplicate messages found.


In [43]:
# ============================================================
# CELL 12 — CHECK FOR CONFLICTING LABELS
# ============================================================

# Check how many different labels are associated with each
# normalized message.
#
# A message should ideally have only one label.
# If the same cleaned message appears as both ham and spam,
# we must investigate it instead of deleting it automatically.

label_counts_per_message = (
    clean_df
    .groupby("clean_message")["label"]
    .nunique()
)

# Select messages associated with more than one label
conflicting_messages = label_counts_per_message[
    label_counts_per_message > 1
]

print("========== CONFLICTING LABEL CHECK ==========\n")

print(
    "Messages with conflicting labels:",
    len(conflicting_messages)
)

if len(conflicting_messages) == 0:

    print("\nNo conflicting labels found.")
    print(
        "Every normalized message has a consistent label."
    )

else:

    print(
        "\nWARNING: Conflicting labels were found."
    )

    print("\nConflicting records:")

    display(
        clean_df[
            clean_df["clean_message"].isin(
                conflicting_messages.index
            )
        ][
            ["label", "message", "clean_message"]
        ].sort_values("clean_message")
    )

========== CONFLICTING LABEL CHECK ==========

Messages with conflicting labels: 0

No conflicting labels found.
Every normalized message has a consistent label.


In [44]:
# ============================================================
# CELL 13 — REMOVE NORMALIZED DUPLICATE MESSAGES
# ============================================================

# Since Cell 12 confirmed that there are no conflicting labels,
# we can safely keep one occurrence of each normalized message.

records_before = len(clean_df)

clean_df = (
    clean_df
    .drop_duplicates(
        subset=["clean_message"],
        keep="first"
    )
    .reset_index(drop=True)
)

records_after = len(clean_df)

print("========== NORMALIZED DEDUPLICATION ==========\n")

print("Records before removal :", records_before)
print("Records after removal  :", records_after)
print("Normalized duplicates removed:",
      records_before - records_after)

# Verify that no normalized duplicates remain
remaining_normalized_duplicates = (
    clean_df["clean_message"].duplicated().sum()
)

print(
    "\nRemaining normalized duplicates:",
    remaining_normalized_duplicates
)

# Verify that no exact duplicate rows remain
print(
    "Remaining exact duplicate rows:",
    clean_df.duplicated().sum()
)

========== NORMALIZED DEDUPLICATION ==========

Records before removal : 5131
Records after removal  : 5131
Normalized duplicates removed: 0

Remaining normalized duplicates: 0
Remaining exact duplicate rows: 0


In [45]:
# ============================================================
# CELL 14 — FINAL CLEAN DATASET VALIDATION
# ============================================================

print("=" * 65)
print("              FINAL CLEAN DATASET VALIDATION")
print("=" * 65)

# Basic dataset information
print("\nDataset shape:")
print(clean_df.shape)

# Check missing values
print("\nMissing values:")
print(clean_df.isnull().sum())

# Check exact duplicate rows
print("\nExact duplicate rows:")
print(clean_df.duplicated().sum())

# Check duplicate normalized messages
print("\nDuplicate normalized messages:")
print(clean_df["clean_message"].duplicated().sum())

# Check empty cleaned messages
empty_count = (
    clean_df["clean_message"].str.len() == 0
).sum()

print("\nEmpty cleaned messages:")
print(empty_count)

# Display final class distribution
print("\nClass distribution:")
print(clean_df["label"].value_counts())

# Display class percentages
class_percentages = (
    clean_df["label"]
    .value_counts(normalize=True)
    * 100
)

print("\nClass percentages:")
print(class_percentages.round(2))

# Verify labels
print("\nUnique labels:")
print(clean_df["label"].unique())

# Final validation conditions
validation_passed = (
    clean_df.isnull().sum().sum() == 0
    and clean_df.duplicated().sum() == 0
    and clean_df["clean_message"].duplicated().sum() == 0
    and empty_count == 0
    and set(clean_df["label"].unique()) == {"ham", "spam"}
)

print("\n" + "=" * 65)

if validation_passed:
    print("FINAL DATASET VALIDATION: PASSED")
    print("The dataset is ready for train/test splitting.")
else:
    print("FINAL DATASET VALIDATION: FAILED")
    print("Do not continue until the issue is resolved.")

print("=" * 65)

              FINAL CLEAN DATASET VALIDATION

Dataset shape:
(5131, 3)

Missing values:
label            0
message          0
clean_message    0
dtype: int64

Exact duplicate rows:
0

Duplicate normalized messages:
0

Empty cleaned messages:
0

Class distribution:
label
ham     4502
spam     629
Name: count, dtype: int64

Class percentages:
label
ham     87.74
spam    12.26
Name: proportion, dtype: float64

Unique labels:
<StringArray>
['ham', 'spam']
Length: 2, dtype: str

FINAL DATASET VALIDATION: PASSED
The dataset is ready for train/test splitting.


In [46]:
# ============================================================
# CELL 15 — CREATE FEATURES AND TRAIN/TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# Separate input features and target labels
# ------------------------------------------------------------

# X = cleaned SMS messages that the model will learn from
X = clean_df["clean_message"]

# y = target labels that the model needs to predict
y = clean_df["label"]

print("========== FEATURES AND TARGET ==========\n")

print("Total samples:", len(X))
print("Features (X) :", X.name)
print("Target (y)   :", y.name)

# ------------------------------------------------------------
# Split into training and testing datasets
# ------------------------------------------------------------

# 80% of the data is used for training.
# 20% is reserved for final evaluation.
#
# stratify=y ensures that the ham/spam ratio remains
# approximately the same in both datasets.
#
# random_state=42 ensures reproducibility.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\n========== TRAIN / TEST SPLIT ==========\n")

print("Total samples    :", len(X))
print("Training samples :", len(X_train))
print("Testing samples  :", len(X_test))

print(
    "\nTraining percentage:",
    round(len(X_train) / len(X) * 100, 2),
    "%"
)

print(
    "Testing percentage :",
    round(len(X_test) / len(X) * 100, 2),
    "%"
)

========== FEATURES AND TARGET ==========

Total samples: 5131
Features (X) : clean_message
Target (y)   : label

========== TRAIN / TEST SPLIT ==========

Total samples    : 5131
Training samples : 4104
Testing samples  : 1027

Training percentage: 79.98 %
Testing percentage : 20.02 %


In [47]:
# ============================================================
# CELL 16 — FINAL TRAIN / TEST VALIDATION
# ============================================================

print("=" * 65)
print("              FINAL TRAIN / TEST VALIDATION")
print("=" * 65)

# ------------------------------------------------------------
# 1. Training class distribution
# ------------------------------------------------------------

print("\n========== TRAINING CLASS DISTRIBUTION ==========\n")

train_counts = y_train.value_counts()
train_percentages = (
    y_train.value_counts(normalize=True) * 100
)

print(train_counts)

print("\nTraining percentages:")
print(train_percentages.round(2))

# ------------------------------------------------------------
# 2. Testing class distribution
# ------------------------------------------------------------

print("\n========== TESTING CLASS DISTRIBUTION ==========\n")

test_counts = y_test.value_counts()
test_percentages = (
    y_test.value_counts(normalize=True) * 100
)

print(test_counts)

print("\nTesting percentages:")
print(test_percentages.round(2))

# ------------------------------------------------------------
# 3. Check for overlap between training and testing data
# ------------------------------------------------------------

training_messages = set(X_train)
testing_messages = set(X_test)

overlap = training_messages.intersection(
    testing_messages
)

print("\n========== DATA LEAKAGE CHECK ==========\n")

print(
    "Overlapping messages:",
    len(overlap)
)

# ------------------------------------------------------------
# 4. Final validation
# ------------------------------------------------------------

print("\n========== VALIDATION RESULT ==========\n")

if len(overlap) == 0:
    print("No overlapping messages found.")
    print("Training and testing datasets are properly separated.")
else:
    print("WARNING: Overlapping messages found.")
    print("Do not proceed to TF-IDF.")

print("\nClass distribution comparison:")

print(
    f"Training  → Ham: {train_percentages['ham']:.2f}% | "
    f"Spam: {train_percentages['spam']:.2f}%"
)

print(
    f"Testing   → Ham: {test_percentages['ham']:.2f}% | "
    f"Spam: {test_percentages['spam']:.2f}%"
)

print("\n" + "=" * 65)

              FINAL TRAIN / TEST VALIDATION

========== TRAINING CLASS DISTRIBUTION ==========

label
ham     3601
spam     503
Name: count, dtype: int64

Training percentages:
label
ham     87.74
spam    12.26
Name: proportion, dtype: float64

========== TESTING CLASS DISTRIBUTION ==========

label
ham     901
spam    126
Name: count, dtype: int64

Testing percentages:
label
ham     87.73
spam    12.27
Name: proportion, dtype: float64

========== DATA LEAKAGE CHECK ==========

Overlapping messages: 0

========== VALIDATION RESULT ==========

No overlapping messages found.
Training and testing datasets are properly separated.

Class distribution comparison:
Training  → Ham: 87.74% | Spam: 12.26%
Testing   → Ham: 87.73% | Spam: 12.27%



In [48]:
# ============================================================
# CELL 17 — INITIALIZE TF-IDF VECTORIZER
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer

# Create the TF-IDF vectorizer.
#
# lowercase=False because our text has already been converted
# to lowercase during preprocessing.
#
# token_pattern is configured to include words containing
# letters and/or numbers.
#
# ngram_range=(1, 2) allows the vectorizer to learn:
#   - unigrams  → individual words
#   - bigrams   → two-word combinations
#
# We will evaluate whether this configuration helps the
# classifier later rather than assuming it is optimal.

tfidf_vectorizer = TfidfVectorizer(
    lowercase=False,
    token_pattern=r"(?u)\b[a-zA-Z0-9]+\b",
    ngram_range=(1, 2)
)

print("TF-IDF vectorizer created successfully.")

print("\nConfiguration:")
print("lowercase   :", tfidf_vectorizer.lowercase)
print("ngram_range :", tfidf_vectorizer.ngram_range)
print("token_pattern:", tfidf_vectorizer.token_pattern)

TF-IDF vectorizer created successfully.

Configuration:
lowercase   : False
ngram_range : (1, 2)
token_pattern: (?u)\b[a-zA-Z0-9]+\b


In [49]:
# ============================================================
# CELL 18 — TF-IDF FIT AND TRANSFORMATION
# ============================================================

# IMPORTANT:
# fit_transform() is used ONLY on the training data.
# This allows the vectorizer to learn its vocabulary and IDF
# values exclusively from the training dataset.
#
# The already-fitted vectorizer is then used with transform()
# on the testing data.

# Fit TF-IDF on training data and transform it
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

# Transform testing data using the SAME fitted vectorizer
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("========== TF-IDF TRANSFORMATION ==========\n")

print("Original training samples :", len(X_train))
print("Original testing samples  :", len(X_test))

print("\nTF-IDF training matrix shape:")
print(X_train_tfidf.shape)

print("\nTF-IDF testing matrix shape:")
print(X_test_tfidf.shape)

print("\nNumber of learned features:",
      len(tfidf_vectorizer.get_feature_names_out()))

print("\nTF-IDF transformation completed successfully.")

========== TF-IDF TRANSFORMATION ==========

Original training samples : 4104
Original testing samples  : 1027

TF-IDF training matrix shape:
(4104, 43856)

TF-IDF testing matrix shape:
(1027, 43856)

Number of learned features: 43856

TF-IDF transformation completed successfully.


In [50]:
# ============================================================
# CELL 19 — INSPECT TF-IDF VOCABULARY
# ============================================================

# Get all features learned from the training data
feature_names = tfidf_vectorizer.get_feature_names_out()

print("========== TF-IDF VOCABULARY ==========\n")

print("Total features:", len(feature_names))

print("\nFirst 30 learned features:")
print(feature_names[:30])

print("\nLast 30 learned features:")
print(feature_names[-30:])

# Display a small sample of features as a DataFrame
feature_sample = pd.DataFrame({
    "feature_index": range(
        min(50, len(feature_names))
    ),
    "feature": feature_names[
        :min(50, len(feature_names))
    ]
})

print("\nSample vocabulary:")
display(feature_sample)

========== TF-IDF VOCABULARY ==========

Total features: 43856

First 30 learned features:
['0' '0 for' '0 key' '00' '00 in' '00 per' '00 sub' '00 subs' '000'
 '000 bonus' '000 cash' '000 homeowners' '000 pounds' '000 price'
 '000 prize' '000 xmas' '000pes' '000pes so' '008704050406'
 '008704050406 sp' '0089' '0089 my' '0121' '0121 2025050' '01223585236'
 '01223585236 xx' '01223585334' '01223585334 to' '0125698789'
 '0125698789 ring']

Last 30 learned features:
['yup not' 'yup ok' 'yup song' 'yup thk' 'yupz' 'yupz i' 'z' 'z will'
 'zac' 'zac doesn' 'zealand' 'zed' 'zed 08701417012' 'zed 08701417012150p'
 'zed pobox' 'zeros' 'zeros that' 'zhong' 'zhong se' 'zindgi' 'zindgi wo'
 'zoe' 'zoe 18' 'zoe it' 'zogtorius' 'zogtorius i' 'zouk' 'zouk with'
 'zyada' 'zyada kisi']

Sample vocabulary:


,feature_index,feature
0,0,0
1,1,0 for
2,2,0 key
3,3,00
4,4,00 in
5,5,00 per
6,6,00 sub
7,7,00 subs
8,8,000
9,9,000 bonus


In [51]:
# ============================================================
# CELL 20 — INSPECT TF-IDF MATRIX
# ============================================================

print("========== TF-IDF MATRIX INSPECTION ==========\n")

# Number of total possible values
train_total_values = (
    X_train_tfidf.shape[0] *
    X_train_tfidf.shape[1]
)

test_total_values = (
    X_test_tfidf.shape[0] *
    X_test_tfidf.shape[1]
)

# Number of non-zero TF-IDF values
train_nonzero_values = X_train_tfidf.nnz
test_nonzero_values = X_test_tfidf.nnz

# Calculate sparsity
train_sparsity = (
    1 - train_nonzero_values / train_total_values
) * 100

test_sparsity = (
    1 - test_nonzero_values / test_total_values
) * 100

print("Training matrix shape :", X_train_tfidf.shape)
print("Testing matrix shape  :", X_test_tfidf.shape)

print("\nTraining non-zero values:", train_nonzero_values)
print("Testing non-zero values :", test_nonzero_values)

print("\nTraining matrix sparsity:",
      round(train_sparsity, 2), "%")

print("Testing matrix sparsity :",
      round(test_sparsity, 2), "%")

# Display the first 10 feature names
print("\nFirst 10 TF-IDF features:")
print(feature_names[:10])

# Inspect the first training message
print("\nFirst training message:")
print(X_train.iloc[0])

# Display its non-zero TF-IDF features
first_row = X_train_tfidf[0]

nonzero_indices = first_row.indices
nonzero_values = first_row.data

first_message_features = pd.DataFrame({
    "feature": feature_names[nonzero_indices],
    "tfidf_value": nonzero_values
}).sort_values(
    "tfidf_value",
    ascending=False
)

print("\nTF-IDF features present in the first message:")
display(first_message_features.head(15))

========== TF-IDF MATRIX INSPECTION ==========

Training matrix shape : (4104, 43856)
Testing matrix shape  : (1027, 43856)

Training non-zero values: 120790
Testing non-zero values : 22313

Training matrix sparsity: 99.93 %
Testing matrix sparsity : 99.95 %

First 10 TF-IDF features:
['0' '0 for' '0 key' '00' '00 in' '00 per' '00 sub' '00 subs' '000'
 '000 bonus']

First training message:
go home liao ask dad to pick me up at 6

TF-IDF features present in the first message:


,feature,tfidf_value
13,liao ask,0.300484
14,ask dad,0.286361
15,dad to,0.286361
12,home liao,0.276341
17,pick me,0.268568
20,at 6,0.262218
11,go home,0.248095
19,up at,0.244425
18,me up,0.244425
16,to pick,0.238075


In [52]:
# ============================================================
# CELL 21 — TRAIN MULTINOMIAL NAIVE BAYES
# ============================================================

from sklearn.naive_bayes import MultinomialNB

# Create the Multinomial Naive Bayes classifier.
#
# alpha controls additive smoothing.
# We start with the standard default value of 1.0.
#
# This is our baseline model. We will evaluate it first
# before considering any tuning.

nb_model = MultinomialNB(
    alpha=1.0
)

# Train the classifier using ONLY the training TF-IDF data
# and corresponding training labels.
nb_model.fit(
    X_train_tfidf,
    y_train
)

print("========== NAIVE BAYES TRAINING ==========\n")

print("Model trained successfully.")
print("Algorithm        : Multinomial Naive Bayes")
print("Alpha (smoothing):", nb_model.alpha)
print("Training samples :", X_train_tfidf.shape[0])
print("Training features:", X_train_tfidf.shape[1])

========== NAIVE BAYES TRAINING ==========

Model trained successfully.
Algorithm        : Multinomial Naive Bayes
Alpha (smoothing): 1.0
Training samples : 4104
Training features: 43856


In [53]:
# ============================================================
# CELL 22 — GENERATE TEST PREDICTIONS
# ============================================================

# Use the trained Naive Bayes model to predict the labels
# of the unseen testing data.

y_pred = nb_model.predict(X_test_tfidf)

print("========== TEST PREDICTION ==========\n")

print("Number of test messages :", len(X_test))
print("Number of predictions   :", len(y_pred))

print("\nFirst 20 actual labels:")
print(y_test.iloc[:20].to_numpy())

print("\nFirst 20 predicted labels:")
print(y_pred[:20])

# Verify that the number of predictions matches
# the number of test samples.
if len(y_pred) == len(y_test):
    print("\nPrediction count matches test-set size.")
else:
    print("\nWARNING: Prediction count does not match test-set size.")

========== TEST PREDICTION ==========

Number of test messages : 1027
Number of predictions   : 1027

First 20 actual labels:
['spam' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'spam' 'ham'
 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham']

First 20 predicted labels:
['ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'spam' 'ham'
 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham']

Prediction count matches test-set size.


In [54]:
# ============================================================
# CELL 23 — BASELINE MODEL EVALUATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# Calculate evaluation metrics
# ------------------------------------------------------------

accuracy = accuracy_score(y_test, y_pred)

# For spam classification, spam is our positive class.
precision = precision_score(
    y_test,
    y_pred,
    pos_label="spam"
)

recall = recall_score(
    y_test,
    y_pred,
    pos_label="spam"
)

f1 = f1_score(
    y_test,
    y_pred,
    pos_label="spam"
)

print("=" * 65)
print("           MULTINOMIAL NAIVE BAYES — BASELINE")
print("=" * 65)

print(f"\nAccuracy  : {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Precision : {precision:.4f} ({precision * 100:.2f}%)")
print(f"Recall    : {recall:.4f} ({recall * 100:.2f}%)")
print(f"F1-Score  : {f1:.4f} ({f1 * 100:.2f}%)")

# ------------------------------------------------------------
# Detailed classification report
# ------------------------------------------------------------

print("\n========== CLASSIFICATION REPORT ==========\n")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["ham", "spam"],
        digits=4
    )
)

# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=["ham", "spam"]
)

print("========== CONFUSION MATRIX ==========\n")

print("                Predicted")
print("                Ham   Spam")
print(f"Actual Ham      {cm[0, 0]:4d}  {cm[0, 1]:4d}")
print(f"Actual Spam     {cm[1, 0]:4d}  {cm[1, 1]:4d}")

print("\nMatrix format:")
print("[[True Ham,  False Spam],")
print(" [False Ham, True Spam ]]")

           MULTINOMIAL NAIVE BAYES — BASELINE

Accuracy  : 0.9318 (93.18%)
Precision : 1.0000 (100.00%)
Recall    : 0.4444 (44.44%)
F1-Score  : 0.6154 (61.54%)

========== CLASSIFICATION REPORT ==========

              precision    recall  f1-score   support

         ham     0.9279    1.0000    0.9626       901
        spam     1.0000    0.4444    0.6154       126

    accuracy                         0.9318      1027
   macro avg     0.9640    0.7222    0.7890      1027
weighted avg     0.9368    0.9318    0.9200      1027

========== CONFUSION MATRIX ==========

                Predicted
                Ham   Spam
Actual Ham       901     0
Actual Spam       70    56

Matrix format:
[[True Ham,  False Spam],
 [False Ham, True Spam ]]


In [55]:
# ============================================================
# CELL 24 — ANALYZE MISSED SPAM MESSAGES
# ============================================================

# Identify messages that were actually spam but predicted as ham.
false_negative_mask = (
    (y_test == "spam") &
    (y_pred == "ham")
)

false_negatives = pd.DataFrame({
    "actual_label": y_test[false_negative_mask],
    "predicted_label": y_pred[false_negative_mask],
    "message": clean_df.loc[
        X_test[false_negative_mask].index,
        "message"
    ].values,
    "clean_message": X_test[false_negative_mask].values
})

print("========== FALSE NEGATIVE ANALYSIS ==========\n")

print(
    "Actual spam messages:",
    (y_test == "spam").sum()
)

print(
    "Spam correctly detected:",
    ((y_test == "spam") & (y_pred == "spam")).sum()
)

print(
    "Spam incorrectly classified as ham:",
    len(false_negatives)
)

print("\nSample missed spam messages:")

display(
    false_negatives.head(20)
)

========== FALSE NEGATIVE ANALYSIS ==========

Actual spam messages: 126
Spam correctly detected: 56
Spam incorrectly classified as ham: 70

Sample missed spam messages:


,actual_label,predicted_label,message,clean_message
2782,spam,ham,We currently have a message awaiting your coll...,we currently have a message awaiting your coll...
2531,spam,ham,Hello darling how are you today? I would love ...,hello darling how are you today i would love t...
415,spam,ham,Someone has contacted our dating service and e...,someone has contacted our dating service and e...
1224,spam,ham,"Hungry gay guys feeling hungry and up 4 it, no...",hungry gay guys feeling hungry and up 4 it now...
1568,spam,ham,U have a secret admirer who is looking 2 make ...,u have a secret admirer who is looking 2 make ...
2293,spam,ham,Babe: U want me dont u baby! Im nasty and have...,babe u want me dont u baby im nasty and have a...
4668,spam,ham,Natalie (20/F) is inviting you to be her frien...,natalie 20 f is inviting you to be her friend ...
1042,spam,ham,Someone U know has asked our dating service 2 ...,someone u know has asked our dating service 2 ...
3924,spam,ham,Missed call alert. These numbers called but le...,missed call alert these numbers called but lef...
3185,spam,ham,Please CALL 08712402972 immediately as there i...,please call 08712402972 immediately as there i...


In [56]:
# ============================================================
# CELL 25 — INSPECT NAIVE BAYES PREDICTION PROBABILITIES
# ============================================================

# Get the probability assigned to each class for every
# test message.
y_test_proba = nb_model.predict_proba(X_test_tfidf)

# Get the class names used internally by the model
model_classes = nb_model.classes_

# Locate the probability column corresponding to spam
spam_index = list(model_classes).index("spam")

# Extract spam probabilities
spam_probabilities = y_test_proba[:, spam_index]

# Create a DataFrame for analysis
probability_analysis = pd.DataFrame({
    "message": X_test.values,
    "actual_label": y_test.values,
    "predicted_label": y_pred,
    "spam_probability": spam_probabilities
})

# Identify false negatives
false_negative_probabilities = probability_analysis[
    (probability_analysis["actual_label"] == "spam") &
    (probability_analysis["predicted_label"] == "ham")
].sort_values(
    "spam_probability",
    ascending=False
)

print("========== NAIVE BAYES PROBABILITY ANALYSIS ==========\n")

print("Model classes:")
print(model_classes)

print("\nSpam probability statistics:")
print(
    pd.Series(spam_probabilities).describe().round(4)
)

print("\nHighest-probability missed spam messages:")

display(
    false_negative_probabilities.head(15)
)

========== NAIVE BAYES PROBABILITY ANALYSIS ==========

Model classes:
['ham' 'spam']

Spam probability statistics:
count    1027.0000
mean        0.0644
std         0.1880
min         0.0000
25%         0.0017
50%         0.0050
75%         0.0156
max         0.9837
dtype: float64

Highest-probability missed spam messages:


,message,actual_label,predicted_label,spam_probability
377,u have a secret admirer reveal who thinks u r ...,spam,ham,0.485489
775,you have been selected to stay in 1 of 250 top...,spam,ham,0.477200
248,you can stop further club tones by replying st...,spam,ham,0.475479
304,new textbuddy chat 2 horny guys in ur area 4 j...,spam,ham,0.473981
1025,txt call to no 86888 claim your reward of 3 ho...,spam,ham,0.469569
867,free ring tone just text polys to 87131 then e...,spam,ham,0.449975
768,do you want a new video phone 600 anytime any ...,spam,ham,0.446885
638,100 dating service cal l 09064012103 box334sk38ch,spam,ham,0.432138
770,good luck draw takes place 28th feb 06 good lu...,spam,ham,0.414869
450,hi 07734396839 ibh customer loyalty offer the ...,spam,ham,0.412644


In [57]:
# ============================================================
# CELL 26 — CROSS-VALIDATED THRESHOLD ANALYSIS
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ------------------------------------------------------------
# CREATE TF-IDF + NAIVE BAYES PIPELINE
# ------------------------------------------------------------

# The vectorizer and classifier are inside one Pipeline.
# This ensures that TF-IDF is fitted independently inside
# every cross-validation fold and prevents data leakage.

cv_pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=False,
            token_pattern=r"(?u)\b[a-zA-Z0-9]+\b",
            ngram_range=(1, 2)
        )
    ),
    (
        "nb",
        MultinomialNB(alpha=1.0)
    )
])

# ------------------------------------------------------------
# STRATIFIED 5-FOLD CROSS-VALIDATION
# ------------------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("========== CROSS-VALIDATED THRESHOLD ANALYSIS ==========\n")

print("Cross-validation folds :", 5)
print("Training samples       :", len(X_train))
print("TF-IDF n-gram range    :", (1, 2))
print("Naive Bayes alpha      :", 1.0)

# ------------------------------------------------------------
# GENERATE OUT-OF-FOLD PROBABILITIES
# ------------------------------------------------------------

# Each training message receives a prediction from a model
# that did NOT train on that particular message.

oof_probabilities = cross_val_predict(
    cv_pipeline,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)

print("\nCross-validation completed successfully.")

print(
    "Out-of-fold probability shape:",
    oof_probabilities.shape
)

# ------------------------------------------------------------
# IDENTIFY THE SPAM PROBABILITY COLUMN
# ------------------------------------------------------------

# MultinomialNB orders the classes alphabetically.
# We determine the correct position directly from y_train
# instead of accessing the unfitted original pipeline.

cv_classes = sorted(
    pd.unique(y_train)
)

print("\nClass order:")
print(cv_classes)

spam_index_cv = cv_classes.index("spam")

# Extract probability of the "spam" class
oof_spam_probability = (
    oof_probabilities[:, spam_index_cv]
)

# ------------------------------------------------------------
# EVALUATE DIFFERENT DECISION THRESHOLDS
# ------------------------------------------------------------

thresholds = [
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50
]

threshold_results = []

for threshold in thresholds:

    # Classify as spam when the predicted spam probability
    # is greater than or equal to the selected threshold.
    oof_prediction = [
        "spam" if probability >= threshold else "ham"
        for probability in oof_spam_probability
    ]

    accuracy_value = accuracy_score(
        y_train,
        oof_prediction
    )

    precision_value = precision_score(
        y_train,
        oof_prediction,
        pos_label="spam",
        zero_division=0
    )

    recall_value = recall_score(
        y_train,
        oof_prediction,
        pos_label="spam",
        zero_division=0
    )

    f1_value = f1_score(
        y_train,
        oof_prediction,
        pos_label="spam",
        zero_division=0
    )

    threshold_results.append({
        "threshold": threshold,
        "accuracy": accuracy_value,
        "precision": precision_value,
        "recall": recall_value,
        "f1_score": f1_value
    })

# Convert results to DataFrame
threshold_results_df = pd.DataFrame(
    threshold_results
)

# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

print("\n========== THRESHOLD PERFORMANCE ==========\n")

display(
    threshold_results_df.round(4)
)

# ------------------------------------------------------------
# SELECT BEST THRESHOLD
# ------------------------------------------------------------

# Primary objective:
# maximize spam F1-score.
#
# If two thresholds have the same F1-score, prefer the one
# with higher accuracy.

best_threshold_row = (
    threshold_results_df
    .sort_values(
        ["f1_score", "accuracy"],
        ascending=False
    )
    .iloc[0]
)

best_threshold = float(
    best_threshold_row["threshold"]
)

print("\n========== BEST CROSS-VALIDATED THRESHOLD ==========\n")

print(
    "Best threshold:",
    best_threshold
)

print(
    "Accuracy :",
    round(
        best_threshold_row["accuracy"] * 100,
        2
    ),
    "%"
)

print(
    "Precision:",
    round(
        best_threshold_row["precision"] * 100,
        2
    ),
    "%"
)

print(
    "Recall   :",
    round(
        best_threshold_row["recall"] * 100,
        2
    ),
    "%"
)

print(
    "F1-Score :",
    round(
        best_threshold_row["f1_score"] * 100,
        2
    ),
    "%"
)

print(
    "\nThe test set was NOT used for threshold selection."
)

========== CROSS-VALIDATED THRESHOLD ANALYSIS ==========

Cross-validation folds : 5
Training samples       : 4104
TF-IDF n-gram range    : (1, 2)
Naive Bayes alpha      : 1.0

Cross-validation completed successfully.
Out-of-fold probability shape: (4104, 2)

Class order:
['ham', 'spam']

========== THRESHOLD PERFORMANCE ==========



,threshold,accuracy,precision,recall,f1_score
0,0.10,0.9710,0.9593,0.7972,0.8708
1,0.15,0.9681,0.9947,0.7435,0.8510
2,0.20,0.9608,0.9942,0.6839,0.8104
3,0.25,0.9547,1.0000,0.6302,0.7732
4,0.30,0.9474,1.0000,0.5706,0.7266
5,0.35,0.9437,1.0000,0.5408,0.7019
6,0.40,0.9384,1.0000,0.4970,0.6640
7,0.45,0.9327,1.0000,0.4513,0.6219
8,0.50,0.9279,1.0000,0.4115,0.5831



========== BEST CROSS-VALIDATED THRESHOLD ==========

Best threshold: 0.1
Accuracy : 97.1 %
Precision: 95.93 %
Recall   : 79.72 %
F1-Score : 87.08 %

The test set was NOT used for threshold selection.


In [58]:
# ============================================================
# CELL 27 — FINAL TEST EVALUATION WITH SELECTED THRESHOLD
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# Use the already-trained baseline Naive Bayes model to obtain
# spam probabilities for the untouched test set.
# ------------------------------------------------------------

test_probabilities = nb_model.predict_proba(
    X_test_tfidf
)

# Find the column corresponding to "spam"
test_classes = nb_model.classes_

spam_index_test = list(
    test_classes
).index("spam")

# Extract spam probability
test_spam_probability = (
    test_probabilities[:, spam_index_test]
)

# ------------------------------------------------------------
# Apply the threshold selected using ONLY training
# cross-validation.
# ------------------------------------------------------------

selected_threshold = best_threshold

y_pred_threshold = [
    "spam"
    if probability >= selected_threshold
    else "ham"
    for probability in test_spam_probability
]

# ------------------------------------------------------------
# Calculate final test metrics
# ------------------------------------------------------------

test_accuracy = accuracy_score(
    y_test,
    y_pred_threshold
)

test_precision = precision_score(
    y_test,
    y_pred_threshold,
    pos_label="spam",
    zero_division=0
)

test_recall = recall_score(
    y_test,
    y_pred_threshold,
    pos_label="spam",
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    y_pred_threshold,
    pos_label="spam",
    zero_division=0
)

print("=" * 65)
print("       NAIVE BAYES — SELECTED THRESHOLD TEST")
print("=" * 65)

print(
    f"\nSelected threshold: {selected_threshold:.2f}"
)

print(
    f"Accuracy  : {test_accuracy:.4f} "
    f"({test_accuracy * 100:.2f}%)"
)

print(
    f"Precision : {test_precision:.4f} "
    f"({test_precision * 100:.2f}%)"
)

print(
    f"Recall    : {test_recall:.4f} "
    f"({test_recall * 100:.2f}%)"
)

print(
    f"F1-Score  : {test_f1:.4f} "
    f"({test_f1 * 100:.2f}%)"
)

# ------------------------------------------------------------
# Detailed classification report
# ------------------------------------------------------------

print("\n========== CLASSIFICATION REPORT ==========\n")

print(
    classification_report(
        y_test,
        y_pred_threshold,
        target_names=["ham", "spam"],
        digits=4
    )
)

# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

cm_threshold = confusion_matrix(
    y_test,
    y_pred_threshold,
    labels=["ham", "spam"]
)

print("========== CONFUSION MATRIX ==========\n")

print("                Predicted")
print("                Ham   Spam")
print(
    f"Actual Ham      {cm_threshold[0, 0]:4d}  "
    f"{cm_threshold[0, 1]:4d}"
)
print(
    f"Actual Spam     {cm_threshold[1, 0]:4d}  "
    f"{cm_threshold[1, 1]:4d}"
)

print("\n" + "=" * 65)
print("Final test evaluation completed.")
print("=" * 65)

       NAIVE BAYES — SELECTED THRESHOLD TEST

Selected threshold: 0.10
Accuracy  : 0.9718 (97.18%)
Precision : 0.9533 (95.33%)
Recall    : 0.8095 (80.95%)
F1-Score  : 0.8755 (87.55%)

========== CLASSIFICATION REPORT ==========

              precision    recall  f1-score   support

         ham     0.9739    0.9945    0.9841       901
        spam     0.9533    0.8095    0.8755       126

    accuracy                         0.9718      1027
   macro avg     0.9636    0.9020    0.9298      1027
weighted avg     0.9714    0.9718    0.9708      1027

========== CONFUSION MATRIX ==========

                Predicted
                Ham   Spam
Actual Ham       896     5
Actual Spam       24   102

Final test evaluation completed.


In [59]:
# ============================================================
# CELL 28 — BASELINE VS OPTIMIZED MODEL COMPARISON
# ============================================================

# Create a comparison table using the two test-set evaluations.

comparison_df = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Spam Precision",
        "Spam Recall",
        "Spam F1-Score"
    ],

    "Baseline (Threshold = 0.50)": [
        accuracy,
        precision,
        recall,
        f1
    ],

    "Optimized (Threshold = 0.10)": [
        test_accuracy,
        test_precision,
        test_recall,
        test_f1
    ]
})

# Calculate improvement
comparison_df["Improvement"] = (
    comparison_df["Optimized (Threshold = 0.10)"]
    - comparison_df["Baseline (Threshold = 0.50)"]
)

print("=" * 75)
print("          BASELINE VS OPTIMIZED MODEL")
print("=" * 75)

display(
    comparison_df.round(4)
)

print("\nPercentage improvement in metrics:")

for _, row in comparison_df.iterrows():

    print(
        f"{row['Metric']:<18} : "
        f"{row['Improvement'] * 100:+.2f} percentage points"
    )

print("\n" + "=" * 75)

          BASELINE VS OPTIMIZED MODEL


,Metric,Baseline (Threshold = 0.50),Optimized (Threshold = 0.10),Improvement
0,Accuracy,0.9318,0.9718,0.0399
1,Spam Precision,1.0000,0.9533,-0.0467
2,Spam Recall,0.4444,0.8095,0.3651
3,Spam F1-Score,0.6154,0.8755,0.2602



Percentage improvement in metrics:
Accuracy           : +3.99 percentage points
Spam Precision     : -4.67 percentage points
Spam Recall        : +36.51 percentage points
Spam F1-Score      : +26.02 percentage points



In [60]:
# ============================================================
# CELL 29 — FINAL MODEL SUMMARY
# ============================================================

print("=" * 70)
print("                 FINAL SPAM CLASSIFIER SUMMARY")
print("=" * 70)

print("\n========== DATASET ==========\n")

print("Original records          :", len(df))
print("After exact deduplication :", 5171)
print("After normalized cleanup  :", len(clean_df))
print("Training samples          :", len(X_train))
print("Testing samples           :", len(X_test))

print("\n========== CLASS DISTRIBUTION ==========\n")

print("Ham messages  :", (clean_df["label"] == "ham").sum())
print("Spam messages :", (clean_df["label"] == "spam").sum())

print("\n========== TEXT REPRESENTATION ==========\n")

print("Technique       : TF-IDF")
print("N-gram range    :", tfidf_vectorizer.ngram_range)
print("Features        :", len(feature_names))
print("Training shape  :", X_train_tfidf.shape)
print("Testing shape   :", X_test_tfidf.shape)

print("\n========== CLASSIFIER ==========\n")

print("Algorithm       : Multinomial Naive Bayes")
print("Alpha           :", nb_model.alpha)
print("Decision        : Probability threshold")
print("Selected threshold:", selected_threshold)

print("\n========== FINAL TEST PERFORMANCE ==========\n")

print(
    f"Accuracy        : {test_accuracy * 100:.2f}%"
)

print(
    f"Spam Precision  : {test_precision * 100:.2f}%"
)

print(
    f"Spam Recall     : {test_recall * 100:.2f}%"
)

print(
    f"Spam F1-Score   : {test_f1 * 100:.2f}%"
)

print("\n========== CONFUSION MATRIX ==========\n")

print("                Predicted")
print("                Ham   Spam")
print(
    f"Actual Ham      {cm_threshold[0, 0]:4d}  "
    f"{cm_threshold[0, 1]:4d}"
)
print(
    f"Actual Spam     {cm_threshold[1, 0]:4d}  "
    f"{cm_threshold[1, 1]:4d}"
)

print("\n" + "=" * 70)
print("                 MODEL DEVELOPMENT COMPLETE")
print("=" * 70)

                 FINAL SPAM CLASSIFIER SUMMARY

========== DATASET ==========

Original records          : 5574
After exact deduplication : 5171
After normalized cleanup  : 5131
Training samples          : 4104
Testing samples           : 1027

========== CLASS DISTRIBUTION ==========

Ham messages  : 4502
Spam messages : 629

========== TEXT REPRESENTATION ==========

Technique       : TF-IDF
N-gram range    : (1, 2)
Features        : 43856
Training shape  : (4104, 43856)
Testing shape   : (1027, 43856)

========== CLASSIFIER ==========

Algorithm       : Multinomial Naive Bayes
Alpha           : 1.0
Decision        : Probability threshold
Selected threshold: 0.1

========== FINAL TEST PERFORMANCE ==========

Accuracy        : 97.18%
Spam Precision  : 95.33%
Spam Recall     : 80.95%
Spam F1-Score   : 87.55%

========== CONFUSION MATRIX ==========

                Predicted
                Ham   Spam
Actual Ham       896     5
Actual Spam       24   102

                 MODEL DEVELOPM

In [61]:
# ============================================================
# CELL 30 — CREATE CUSTOM SMS SPAM CLASSIFIER
# ============================================================

def classify_sms(message):
    """
    Classify a new SMS message as HAM or SPAM.

    The message follows the same preprocessing and TF-IDF
    transformation used during model development.
    """

    # --------------------------------------------------------
    # Step 1: Validate input
    # --------------------------------------------------------

    if not isinstance(message, str):
        raise TypeError(
            "Message must be provided as a string."
        )

    if not message.strip():
        raise ValueError(
            "Message cannot be empty."
        )

    # --------------------------------------------------------
    # Step 2: Apply the same preprocessing used during training
    # --------------------------------------------------------

    cleaned_message = preprocess_text(message)

    # --------------------------------------------------------
    # Step 3: Convert the cleaned message into TF-IDF features
    #
    # IMPORTANT:
    # We use transform(), NOT fit_transform().
    # The vectorizer was already fitted on training data.
    # --------------------------------------------------------

    message_tfidf = tfidf_vectorizer.transform(
        [cleaned_message]
    )

    # --------------------------------------------------------
    # Step 4: Get spam probability
    # --------------------------------------------------------

    probabilities = nb_model.predict_proba(
        message_tfidf
    )

    spam_index = list(
        nb_model.classes_
    ).index("spam")

    spam_probability = probabilities[
        0,
        spam_index
    ]

    # --------------------------------------------------------
    # Step 5: Apply the cross-validated threshold
    # --------------------------------------------------------

    predicted_label = (
        "spam"
        if spam_probability >= selected_threshold
        else "ham"
    )

    return {
        "original_message": message,
        "cleaned_message": cleaned_message,
        "prediction": predicted_label,
        "spam_probability": spam_probability
    }


print("Custom SMS classifier created successfully.")
print(
    "Decision threshold:",
    selected_threshold
)

Custom SMS classifier created successfully.
Decision threshold: 0.1


In [62]:
# ============================================================
# CELL 31 — TEST CUSTOM SMS CLASSIFIER
# ============================================================

# Create new SMS messages that were not used during training.
test_messages = [
    # HAM examples
    "Hey, are we still meeting for lunch today?",
    "Mom asked me to call her when I reach home.",
    "I will be late by 10 minutes. Please wait for me.",
    
    # SPAM examples
    "Congratulations! You have won a cash prize of 50000. Call now to claim your reward.",
    "URGENT! You have been selected for a FREE gift voucher. Reply WIN to claim now.",
    "You have won a brand new mobile phone. Call 9876543210 immediately to receive your prize."
]

print("=" * 80)
print("                  CUSTOM SMS CLASSIFIER TEST")
print("=" * 80)

for index, message in enumerate(test_messages, start=1):

    result = classify_sms(message)

    print(f"\nTest Message {index}")
    print("-" * 80)

    print("Original message:")
    print(result["original_message"])

    print("\nCleaned message:")
    print(result["cleaned_message"])

    print("\nPrediction:")
    print(result["prediction"].upper())

    print(
        "\nSpam probability:",
        f"{result['spam_probability']:.4f}"
    )

    print(
        "Decision threshold:",
        f"{selected_threshold:.2f}"
    )

    print("-" * 80)

                  CUSTOM SMS CLASSIFIER TEST

Test Message 1
--------------------------------------------------------------------------------
Original message:
Hey, are we still meeting for lunch today?

Cleaned message:
hey are we still meeting for lunch today

Prediction:
HAM

Spam probability: 0.0058
Decision threshold: 0.10
--------------------------------------------------------------------------------

Test Message 2
--------------------------------------------------------------------------------
Original message:
Mom asked me to call her when I reach home.

Cleaned message:
mom asked me to call her when i reach home

Prediction:
HAM

Spam probability: 0.0035
Decision threshold: 0.10
--------------------------------------------------------------------------------

Test Message 3
--------------------------------------------------------------------------------
Original message:
I will be late by 10 minutes. Please wait for me.

Cleaned message:
i will be late by 10 minutes please w

In [63]:
# ============================================================
# CELL 33 — SAVE FINAL SPAM CLASSIFIER
# ============================================================

from pathlib import Path
import joblib

# ------------------------------------------------------------
# Identify the Week-3 project root
# ------------------------------------------------------------

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

# Project-level models directory
MODELS_DIR = PROJECT_ROOT / "models"

# Create the models directory if it does not exist
MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Save the fitted TF-IDF vectorizer
# ------------------------------------------------------------

tfidf_path = MODELS_DIR / "tfidf_vectorizer.joblib"

joblib.dump(
    tfidf_vectorizer,
    tfidf_path
)

# ------------------------------------------------------------
# Save the trained Multinomial Naive Bayes model
# ------------------------------------------------------------

model_path = MODELS_DIR / "spam_classifier.joblib"

joblib.dump(
    nb_model,
    model_path
)

# ------------------------------------------------------------
# Save the selected decision threshold
# ------------------------------------------------------------

threshold_path = MODELS_DIR / "decision_threshold.joblib"

joblib.dump(
    selected_threshold,
    threshold_path
)

# ------------------------------------------------------------
# Verify that all files were created
# ------------------------------------------------------------

print("=" * 70)
print("              MODEL SAVING COMPLETED")
print("=" * 70)

print("\nModels directory:")
print(MODELS_DIR)

print("\nSaved files:")

for path in [
    tfidf_path,
    model_path,
    threshold_path
]:

    if path.exists():

        file_size = path.stat().st_size

        print(
            f"✓ {path.name} "
            f"({file_size / 1024:.2f} KB)"
        )

    else:

        print(
            f"✗ {path.name} — FILE NOT FOUND"
        )

print("\nAll model components saved successfully.")

              MODEL SAVING COMPLETED

Models directory:
C:\Users\ashok\OneDrive\Desktop\EDP_AIML_Internship\WEEK - 3\models

Saved files:
✓ tfidf_vectorizer.joblib (976.68 KB)
✓ spam_classifier.joblib (1371.29 KB)
✓ decision_threshold.joblib (0.02 KB)

All model components saved successfully.


In [64]:
# ============================================================
# CELL 41 — FINAL REPRODUCIBILITY VERIFICATION
# ============================================================

import joblib
import numpy as np

print("=" * 70)
print("             FINAL REPRODUCIBILITY TEST")
print("=" * 70)

# ------------------------------------------------------------
# Load the final saved artifacts from the project-level
# models directory.
# ------------------------------------------------------------

reloaded_tfidf = joblib.load(
    MODELS_DIR / "tfidf_vectorizer.joblib"
)

reloaded_model = joblib.load(
    MODELS_DIR / "spam_classifier.joblib"
)

reloaded_threshold = joblib.load(
    MODELS_DIR / "decision_threshold.joblib"
)

print("\nSaved artifacts loaded successfully.")

# ------------------------------------------------------------
# Verify model configuration
# ------------------------------------------------------------

print("\n========== CONFIGURATION VERIFICATION ==========\n")

print(
    "TF-IDF features:",
    len(
        reloaded_tfidf.get_feature_names_out()
    )
)

print(
    "TF-IDF n-gram range:",
    reloaded_tfidf.ngram_range
)

print(
    "Naive Bayes alpha:",
    reloaded_model.alpha
)

print(
    "Decision threshold:",
    reloaded_threshold
)

# ------------------------------------------------------------
# Test messages
# ------------------------------------------------------------

verification_messages = [
    "Hey, I will reach home by 8 tonight.",
    "Congratulations! You have won a free cash prize. Call now!"
]

print("\n========== REPRODUCIBILITY PREDICTIONS ==========\n")

verification_passed = True

for index, message in enumerate(
    verification_messages,
    start=1
):

    # Apply the same preprocessing
    cleaned_message = preprocess_text(
        message
    )

    # Transform using the reloaded vectorizer
    message_tfidf = reloaded_tfidf.transform(
        [cleaned_message]
    )

    # Generate probabilities using the reloaded model
    probabilities = reloaded_model.predict_proba(
        message_tfidf
    )

    spam_index = list(
        reloaded_model.classes_
    ).index("spam")

    spam_probability = probabilities[
        0,
        spam_index
    ]

    # Apply the reloaded threshold
    prediction = (
        "spam"
        if spam_probability >= reloaded_threshold
        else "ham"
    )

    print(f"Message {index}:")
    print(message)

    print(
        "Prediction:",
        prediction.upper()
    )

    print(
        "Spam probability:",
        f"{spam_probability:.4f}"
    )

    print()

# ------------------------------------------------------------
# Verify the final model configuration
# ------------------------------------------------------------

configuration_valid = (
    len(
        reloaded_tfidf.get_feature_names_out()
    ) == 43856
    and
    reloaded_tfidf.ngram_range == (1, 2)
    and
    reloaded_model.alpha == 1.0
    and
    reloaded_threshold == 0.1
)

print("=" * 70)

if configuration_valid:
    print("FINAL REPRODUCIBILITY TEST: PASSED")
    print(
        "Saved artifacts reproduce the expected "
        "classifier configuration."
    )
else:
    print("FINAL REPRODUCIBILITY TEST: FAILED")

print("=" * 70)

             FINAL REPRODUCIBILITY TEST

Saved artifacts loaded successfully.

========== CONFIGURATION VERIFICATION ==========

TF-IDF features: 43856
TF-IDF n-gram range: (1, 2)
Naive Bayes alpha: 1.0
Decision threshold: 0.1

========== REPRODUCIBILITY PREDICTIONS ==========

Message 1:
Hey, I will reach home by 8 tonight.
Prediction: HAM
Spam probability: 0.0037

Message 2:
Congratulations! You have won a free cash prize. Call now!
Prediction: SPAM
Spam probability: 0.7604

FINAL REPRODUCIBILITY TEST: PASSED
Saved artifacts reproduce the expected classifier configuration.
